# CS781 Phase 1 — Notebook 2: Encode the corpus into token shards

Tokenizing on the fly during training would waste GPU time, so we tokenize **once** and store token ids as raw
`uint16` binary files ("shards"). Training then reads random windows straight from disk with `np.memmap`
(the OS pages in only the bytes we touch, so RAM use stays tiny even for billions of tokens).

* 1 token = 2 bytes (vocab < 65,536) → 1B tokens ≈ 2 GB on disk.
* every document is followed by `<|endoftext|>` (id 0), so the model learns document boundaries
  (the starter concatenated documents with no separator).
* cleaning, de-duplication and the train/val split are **identical** to Notebook 1 (same shared functions,
  same duplicate lists), so held-out evaluation docs are guaranteed never to appear in training shards.
* output: `shards/<tokenizer>/<source>_<train|val>_<k>.bin` + `shards/<tokenizer>/summary.json`

**Resuming**: progress (documents done, next shard number) is saved after every shard. If the run crashes or the
time budget runs out, re-run with this notebook's previous output attached as input; finished shards are kept and
the stream skips already-encoded documents.

**Inputs to attach on Kaggle**: output of Notebook 1 (needs `data/` and `tokenizer/`), and — when resuming — the
previous version of this notebook. Accelerator: None (CPU). Internet: ON.

In [1]:
!pip -q install ijson tokenizers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 3.2 MB/s eta 0:00:00


In [2]:
# ============================================================================================
# SHARED SETUP  —  paste this SAME cell at the top of every Phase-1 notebook.
# Works on Kaggle (Save & Commit or interactive), Colab (Drive) and local machines.
# ============================================================================================
import os, sys, json, time, glob, shutil, random, math, re, zlib, hashlib, gc
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

SESSION_START = time.time()
TIME_BUDGET_H = 11.0   # Kaggle kills a session at 12h and then saves NOTHING -> every long loop stops itself before this

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
PROJECT   = "cs781_p1"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, SCRATCH = Path("/content/drive/MyDrive") / PROJECT, Path("/content/scratch")   # Drive = persistent
elif IN_KAGGLE:
    ROOT, SCRATCH = Path("/kaggle/working") / PROJECT, Path("/tmp/cs781_scratch")       # working = saved on commit
else:
    ROOT, SCRATCH = Path("./") / PROJECT, Path("./scratch")
ROOT    = Path(os.environ.get("CS781_ROOT", ROOT))
SCRATCH = Path(os.environ.get("CS781_SCRATCH", SCRATCH))
ROOT.mkdir(parents=True, exist_ok=True); SCRATCH.mkdir(parents=True, exist_ok=True)

ENV = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
print(f"env={ENV}  ROOT={ROOT}  SCRATCH={SCRATCH}")
for p in {ROOT, SCRATCH}:
    du = shutil.disk_usage(p); print(f"  disk @ {p}: free {du.free/1e9:.1f} GB / total {du.total/1e9:.1f} GB")


# ---------------------------------------------------------------- small utilities
def time_left_h():
    return TIME_BUDGET_H - (time.time() - SESSION_START) / 3600

def save_json(obj, path):
    """Atomic write: a crash mid-write can never leave a half-written (corrupt) state file."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, path)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists(): return default
    with open(path, encoding="utf-8") as f: return json.load(f)

def restore_from_kaggle_inputs(subdirs):
    """Kaggle 'Save & Commit' starts every run with an EMPTY /kaggle/working.
    To resume, attach the output of the previous version (Add Input -> Your Work -> Notebook output).
    This copies <PROJECT>/<subdir> from any attached input into ROOT (never overwriting newer local files)."""
    if not IN_KAGGLE: return
    for sub in subdirs:
        for cand in sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{sub}", recursive=True)):
            n = 0
            for src in Path(cand).rglob("*"):
                if src.is_file():
                    dst = ROOT / sub / src.relative_to(cand)
                    if not dst.exists():
                        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); n += 1
            print(f"restored {n} files: {cand} -> {ROOT/sub}")

def find_artifact(rel):
    """Look for ROOT/rel first, then inside any attached Kaggle input."""
    p = ROOT / rel
    if p.exists(): return p
    if IN_KAGGLE:
        hits = sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{rel}", recursive=True))
        if hits: return Path(hits[0])
    return None

def get_hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("HF")
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get("HF")
    except Exception as e:
        print("Could not read secret 'HF':", e)
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN
print("HF token found:", bool(HF_TOKEN))

# Optional: a PRIVATE HF dataset repo used only to move YOUR OWN artifacts between accounts/sessions.
HF_ARTIFACT_REPO = None   # e.g. "your-hf-username/cs781-p1-artifacts"

def hf_push(subdir):
    if not HF_ARTIFACT_REPO: print("HF_ARTIFACT_REPO not set; skip push"); return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_ARTIFACT_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(ROOT / subdir), path_in_repo=subdir,
                      repo_id=HF_ARTIFACT_REPO, repo_type="dataset")
    print("pushed", subdir)

def hf_pull(subdir):
    if not HF_ARTIFACT_REPO: return
    from huggingface_hub import snapshot_download
    snapshot_download(HF_ARTIFACT_REPO, repo_type="dataset", allow_patterns=[f"{subdir}/*", f"{subdir}/**"],
                      local_dir=str(ROOT), token=HF_TOKEN)
    print("pulled", subdir)


# ---------------------------------------------------------------- data utilities
# These MUST be identical in every notebook (the train/val split and cleaning depend on them).
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SOURCE_NAMES = ["acts", "high_courts", "supreme_court", "tribunals"]   # processing order (matters for dedup)
VAL_PER_10K  = 100            # 1% of documents of every source are held out (document-level split)
PROMPT_WORDS, CONT_WORDS = 512, 128

def extract_text(rec):
    if rec is None: return ""
    if isinstance(rec, str): return rec
    if isinstance(rec, list): return "\n".join(extract_text(r) for r in rec)
    if isinstance(rec, dict):
        for k in ("text", "content", "document", "judgment", "judgement", "body"):
            if k in rec: return extract_text(rec[k])
    return ""

def iter_docs(path):
    """Stream documents from a (possibly multi-GB) JSON file WITHOUT loading it all into RAM.
    Handles: [ {...}, ... ]   |   { id: {...}, ... }   |   JSON-lines."""
    import ijson
    with open(path, "rb") as f:
        head = f.read(1 << 16).lstrip()
        f.seek(0)
        if head[:1] == b"[":
            for rec in ijson.items(f, "item", use_float=True):
                yield extract_text(rec)
            return
        first_line = f.readline(50_000_000); second = f.read(1 << 10).lstrip()[:1]
        f.seek(0)
        is_jsonl = False
        if first_line.endswith(b"\n") and second == b"{":
            try: json.loads(first_line); is_jsonl = True
            except Exception: pass
        if is_jsonl:
            for line in f:
                if line.strip(): yield extract_text(json.loads(line))
        else:
            for _, rec in ijson.kvitems(f, "", use_float=True):
                yield extract_text(rec)

_ZW = re.compile(r"[​‌‍﻿­]")
_WS_ALL = re.compile(r"\s+")
_HSPACE = re.compile(r"[^\S\n]+")
_MANY_NL = re.compile(r"\n\s*\n\s*\n+")

def normalize(text, mode):
    """'collapse_all'  -> every whitespace run becomes one space (text looks like ' '.join(text.split()))
       'keep_newlines' -> collapse spaces/tabs, keep at most one blank line"""
    text = _ZW.sub("", text)
    if mode == "collapse_all":
        return _WS_ALL.sub(" ", text).strip()
    text = _HSPACE.sub(" ", text.replace("\r\n", "\n").replace("\r", "\n"))
    text = "\n".join(l.strip() for l in text.split("\n"))
    return _MANY_NL.sub("\n\n", text).strip()

def hash64(text):
    return int.from_bytes(hashlib.blake2b(text.encode("utf-8", "ignore"), digest_size=8).digest(), "little")

def doc_split(source, idx):
    return "val" if zlib.crc32(f"{source}:{idx}".encode()) % 10000 < VAL_PER_10K else "train"

def list_phase1_files(local_raw_dir=None):
    """Return {'train': {source: repo_path}, 'test': repo_path, 'sizes': {repo_path: bytes}}."""
    if local_raw_dir:
        files = [str(Path(p).relative_to(local_raw_dir)) for p in glob.glob(f"{local_raw_dir}/**/*.json*", recursive=True)]
        sizes = {f: os.path.getsize(Path(local_raw_dir) / f) for f in files}
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        try:
            entries = list(api.list_repo_tree(HF_DATASET_REPO, repo_type="dataset",
                                              path_in_repo="Phase_1_Pretraining", recursive=True))
            files = [e.path for e in entries if getattr(e, "size", None) is not None]
            sizes = {e.path: e.size for e in entries if getattr(e, "size", None) is not None}
        except Exception as e:
            print("list_repo_tree failed (", e, ") -> falling back to list_repo_files")
            files = [f for f in api.list_repo_files(HF_DATASET_REPO, repo_type="dataset") if f.startswith("Phase_1")]
            sizes = {}
    out = {"train": {}, "test": None, "sizes": sizes}
    for f in files:
        name = Path(f).name.lower()
        if not name.endswith((".json", ".jsonl")): continue
        if "test" in name: out["test"] = f
        for s in SOURCE_NAMES:
            if name.startswith("train") and s in name: out["train"][s] = f
    return out

def get_raw_file(repo_path, local_raw_dir=None):
    if local_raw_dir:
        return Path(local_raw_dir) / repo_path
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, repo_path, repo_type="dataset", token=HF_TOKEN,
                                local_dir=str(SCRATCH / "raw")))

def drop_raw_file(path, local_raw_dir=None):
    if local_raw_dir: return          # never delete user-provided files
    try: os.remove(path); print("deleted raw", path)
    except OSError: pass

env=kaggle  ROOT=/kaggle/working/cs781_p1  SCRATCH=/tmp/cs781_scratch
  disk @ /tmp/cs781_scratch: free 1201.7 GB / total 8758.4 GB
  disk @ /kaggle/working/cs781_p1: free 20.9 GB / total 21.0 GB
HF token found: True


## 1. Configuration

In [3]:
LOCAL_RAW_DIR   = os.environ.get("CS781_LOCAL_RAW")
TOKENIZER_NAME  = "bpe32768"          # which tokenizer from Notebook 1 to use
SHARD_TOKENS    = 100_000_000         # tokens per train shard (200 MB)
ENCODE_BATCH    = 512                 # documents per encode_batch call (multi-threaded in the library)
# Fraction of TRAIN documents to keep per source (document-level random subsample, unbiased w.r.t. file order).
# 1.0 = everything. Lower it if disk is tight or for quick ablation shards.
KEEP_FRACTION   = {"acts": 1.0, "high_courts": 1.0, "supreme_court": 1.0, "tribunals": 1.0}

restore_from_kaggle_inputs(["data", "tokenizer", f"shards/{TOKENIZER_NAME}"])
hf_pull("data"); hf_pull("tokenizer")

decisions = load_json(find_artifact("data/decisions.json"))
assert decisions, "data/decisions.json not found — attach Notebook 1's output"
NORM  = decisions["normalize"]
FILES = load_json(find_artifact("data/files.json"))
tok_path = find_artifact(f"tokenizer/{TOKENIZER_NAME}/tokenizer.json")
assert tok_path, f"tokenizer {TOKENIZER_NAME} not found"

from tokenizers import Tokenizer
tok = Tokenizer.from_file(str(tok_path))
EOS = tok.token_to_id("<|endoftext|>")
assert tok.get_vocab_size() < 65536 and EOS == 0
OUT = ROOT / "shards" / TOKENIZER_NAME
OUT.mkdir(parents=True, exist_ok=True)
(ROOT / "tokenizer" / TOKENIZER_NAME).mkdir(parents=True, exist_ok=True)
if not (ROOT / "tokenizer" / TOKENIZER_NAME / "tokenizer.json").exists():
    shutil.copy2(tok_path, ROOT / "tokenizer" / TOKENIZER_NAME / "tokenizer.json")   # keep a copy next to shards
print("normalisation:", NORM, "| tokenizer:", TOKENIZER_NAME, "vocab", tok.get_vocab_size())

restored 20 files: /kaggle/input/notebooks/aaravoswal/01-data-tokenizer-evalset/cs781_p1/data -> /kaggle/working/cs781_p1/data
restored 5 files: /kaggle/input/notebooks/aaravoswal/01-data-tokenizer-evalset/cs781_p1/tokenizer -> /kaggle/working/cs781_p1/tokenizer
normalisation: collapse_all | tokenizer: bpe32768 vocab 32768


## 2. Size estimate (does it fit on disk?)

In [4]:
rep = {r["name"]: r for r in load_json(find_artifact("tokenizer/tokenizer_report.json"), [])}
est_total = 0
for s in SOURCE_NAMES:
    st = load_json(find_artifact(f"data/{s}_stats.json"))
    fert = rep.get(TOKENIZER_NAME, {}).get(f"fertility_{s}", 1.3)
    est = st["words"] * fert * KEEP_FRACTION[s] + st["docs"]
    est_total += est
    print(f"{s:14s} ~{est/1e9:6.2f} B tokens  ~{2*est/1e9:6.2f} GB")
free = shutil.disk_usage(ROOT).free
print(f"TOTAL ~{est_total/1e9:.2f} B tokens  ~{2*est_total/1e9:.1f} GB;  free at ROOT: {free/1e9:.1f} GB")
if 2 * est_total > 0.9 * free:
    print("WARNING: shards may not fit — lower KEEP_FRACTION for the largest source(s).")
if IN_KAGGLE and 2 * est_total > 18e9:
    print("WARNING: Kaggle saves at most ~20 GB of /kaggle/working output — lower KEEP_FRACTION.")

acts           ~  0.15 B tokens  ~  0.29 GB
high_courts    ~  1.53 B tokens  ~  3.05 GB
supreme_court  ~  0.25 B tokens  ~  0.51 GB
tribunals      ~  1.43 B tokens  ~  2.87 GB
TOTAL ~3.36 B tokens  ~6.7 GB;  free at ROOT: 20.9 GB


## 3. Encode
For each document (in file order): clean → skip empty / duplicate → train or val (same hash rule as NB1) →
(train only) keep with probability `KEEP_FRACTION` → `encode_batch` → append ids + `<|endoftext|>`.
When the train buffer reaches `SHARD_TOKENS`, both buffers are written to disk and the state file is updated.

In [5]:
def keep_doc(source, idx):
    return zlib.crc32(f"keep:{source}:{idx}".encode()) % 10000 < KEEP_FRACTION[source] * 10000

def encode_texts(texts):
    enc = tok.encode_batch_fast(texts) if hasattr(tok, "encode_batch_fast") else tok.encode_batch(texts)
    return [np.array(e.ids, dtype=np.uint16) for e in enc]

def write_bin(arr, path):
    tmp = path.with_suffix(".tmp")
    arr.tofile(tmp); os.replace(tmp, path)

STOP = False
state_path = OUT / "encode_state.json"
state = load_json(state_path, {})
for source in SOURCE_NAMES:
    st = state.get(source, {"docs_done": 0, "shard": 0, "finished": False})
    if st["finished"]:
        print(f"[{source}] finished earlier — skip"); continue
    if time_left_h() < 0.5:
        STOP = True; break
    dups = set(np.load(find_artifact(f"data/{source}_dups.npy")).tolist())
    raw_path = get_raw_file(FILES["train"][source], LOCAL_RAW_DIR)
    print(f"[{source}] resuming at doc {st['docs_done']:,}, shard {st['shard']}")
    t0 = time.time(); n_tok = 0
    batch_texts, batch_split = [], []
    train_buf, val_buf, train_len = [], [], 0

    def flush(final=False):
        global train_len
        if train_buf:
            write_bin(np.concatenate(train_buf), OUT / f"{source}_train_{st['shard']:04d}.bin")
        if val_buf:
            write_bin(np.concatenate(val_buf), OUT / f"{source}_val_{st['shard']:04d}.bin")
        if train_buf or val_buf: st["shard"] += 1
        train_buf.clear(); val_buf.clear(); train_len = 0
        st["finished"] = final
        state[source] = st; save_json(state, state_path)

    def run_batch(last_idx):
        global train_len, n_tok
        for ids, sp in zip(encode_texts(batch_texts), batch_split):
            ids = np.append(ids, np.uint16(EOS))
            (train_buf if sp == "train" else val_buf).append(ids)
            n_tok += len(ids)
            if sp == "train": train_len += len(ids)
        batch_texts.clear(); batch_split.clear()
        st["docs_done"] = last_idx + 1      # only becomes durable when flush() saves the state

    idx = -1
    for idx, raw in enumerate(iter_docs(raw_path)):
        if idx < st["docs_done"]: continue                # already encoded in an earlier run
        if idx in dups: continue
        text = normalize(raw, NORM)
        if not text: continue
        sp = doc_split(source, idx)
        if sp == "train" and not keep_doc(source, idx): continue
        batch_texts.append(text); batch_split.append(sp)
        if len(batch_texts) >= ENCODE_BATCH:
            run_batch(idx)
            if train_len >= SHARD_TOKENS:
                flush()
                el = time.time() - t0
                print(f"  [{source}] shard {st['shard']-1} written | doc {idx:,} | {n_tok/1e6:,.0f}M tok this run "
                      f"| {n_tok/el/1e6:.2f}M tok/s | {time_left_h():.1f}h left")
                if time_left_h() < 0.25:
                    STOP = True; break
    if STOP:
        print(f"[{source}] stopped early to stay inside the session budget — re-run to continue"); break
    if batch_texts: run_batch(idx)
    flush(final=True)
    print(f"[{source}] done in {(time.time()-t0)/60:.1f} min")
    drop_raw_file(raw_path, LOCAL_RAW_DIR)
    gc.collect()

ALL_DONE = all(state.get(s, {}).get("finished") for s in SOURCE_NAMES)
print("ALL SOURCES ENCODED:", ALL_DONE)

Phase_1_Pretraining/Train/train_acts.jso(…):   0%|          | 0.00/729M [00:00<?, ?B/s]

[acts] resuming at doc 0, shard 0
  [acts] shard 0 written | doc 12,297 | 102M tok this run | 0.69M tok/s | 11.0h left
[acts] done in 3.6 min
deleted raw /tmp/cs781_scratch/raw/Phase_1_Pretraining/Train/train_acts.json


Phase_1_Pretraining/Train/train_high_cou(…):   0%|          | 0.00/10.7G [00:00<?, ?B/s]

[high_courts] resuming at doc 0, shard 0
  [high_courts] shard 0 written | doc 190,464 | 101M tok this run | 0.65M tok/s | 10.9h left
  [high_courts] shard 1 written | doc 392,640 | 202M tok this run | 0.68M tok/s | 10.8h left
  [high_courts] shard 2 written | doc 603,130 | 303M tok this run | 0.69M tok/s | 10.8h left
  [high_courts] shard 3 written | doc 731,409 | 405M tok this run | 0.70M tok/s | 10.8h left
  [high_courts] shard 4 written | doc 834,705 | 507M tok this run | 0.70M tok/s | 10.7h left
  [high_courts] shard 5 written | doc 907,968 | 608M tok this run | 0.70M tok/s | 10.7h left
  [high_courts] shard 6 written | doc 1,054,648 | 709M tok this run | 0.70M tok/s | 10.6h left
  [high_courts] shard 7 written | doc 1,225,989 | 810M tok this run | 0.70M tok/s | 10.6h left
  [high_courts] shard 8 written | doc 1,356,349 | 912M tok this run | 0.71M tok/s | 10.6h left
  [high_courts] shard 9 written | doc 1,520,949 | 1,013M tok this run | 0.71M tok/s | 10.5h left
  [high_courts] sha

Phase_1_Pretraining/Train/train_supreme_(…):   0%|          | 0.00/1.32G [00:00<?, ?B/s]

[supreme_court] resuming at doc 0, shard 0
  [supreme_court] shard 0 written | doc 22,593 | 102M tok this run | 0.69M tok/s | 10.3h left
  [supreme_court] shard 1 written | doc 43,668 | 206M tok this run | 0.66M tok/s | 10.2h left
[supreme_court] done in 6.4 min
deleted raw /tmp/cs781_scratch/raw/Phase_1_Pretraining/Train/train_supreme_court.json


Phase_1_Pretraining/Train/train_tribunal(…):   0%|          | 0.00/6.48G [00:00<?, ?B/s]

[tribunals] resuming at doc 0, shard 0
  [tribunals] shard 0 written | doc 43,007 | 102M tok this run | 0.72M tok/s | 10.1h left
  [tribunals] shard 1 written | doc 93,696 | 203M tok this run | 0.73M tok/s | 10.1h left
  [tribunals] shard 2 written | doc 140,800 | 304M tok this run | 0.72M tok/s | 10.1h left
  [tribunals] shard 3 written | doc 247,818 | 405M tok this run | 0.73M tok/s | 10.0h left
  [tribunals] shard 4 written | doc 355,340 | 507M tok this run | 0.73M tok/s | 10.0h left
  [tribunals] shard 5 written | doc 417,867 | 608M tok this run | 0.73M tok/s | 10.0h left
  [tribunals] shard 6 written | doc 467,928 | 710M tok this run | 0.74M tok/s | 9.9h left
  [tribunals] shard 7 written | doc 517,391 | 811M tok this run | 0.74M tok/s | 9.9h left
  [tribunals] shard 8 written | doc 565,231 | 912M tok this run | 0.74M tok/s | 9.8h left
  [tribunals] shard 9 written | doc 608,456 | 1,014M tok this run | 0.74M tok/s | 9.8h left
  [tribunals] shard 10 written | doc 658,942 | 1,115M t

## 4. Summary and sanity checks
Token counts per source/split (these numbers are the basis for data-mixing weights), plus a decoded window from a
random shard to confirm the bytes on disk really are the text we expect.

In [6]:
summary = {"tokenizer": TOKENIZER_NAME, "normalize": NORM, "eos_id": EOS, "dtype": "uint16",
           "keep_fraction": KEEP_FRACTION, "complete": ALL_DONE, "shards": [], "totals": {}}
for p in sorted(OUT.glob("*.bin")):
    src, split, _ = p.stem.rsplit("_", 2)
    n = p.stat().st_size // 2
    summary["shards"].append({"file": p.name, "source": src, "split": split, "tokens": n})
    summary["totals"].setdefault(f"{src}/{split}", 0)
    summary["totals"][f"{src}/{split}"] += n
save_json(summary, OUT / "summary.json")
tot_train = sum(v for k, v in summary["totals"].items() if k.endswith("/train"))
for k, v in sorted(summary["totals"].items()):
    share = f"{100*v/tot_train:5.1f}% of train" if k.endswith("/train") and tot_train else ""
    print(f"{k:26s} {v/1e6:10.1f}M tokens  {share}")
print(f"TOTAL train tokens: {tot_train/1e9:.3f} B   ({2*tot_train/1e9:.1f} GB)")

train_shards = [s for s in summary["shards"] if s["split"] == "train"]
if train_shards:
    rng = np.random.default_rng(0)
    sh = train_shards[rng.integers(len(train_shards))]
    mm = np.memmap(OUT / sh["file"], dtype=np.uint16, mode="r")
    i = int(rng.integers(0, max(1, len(mm) - 300)))
    print(f"\n--- decoded sample from {sh['file']} @ {i} ---\n", tok.decode(mm[i:i + 300].tolist(), skip_special_tokens=False))
    print("\nEOS count in this shard (≈ #documents):", int((np.asarray(mm) == EOS).sum()))

acts/train                      148.8M tokens    4.4% of train
acts/val                          1.1M tokens  
high_courts/train              1577.4M tokens   46.3% of train
high_courts/val                  16.0M tokens  
supreme_court/train             253.2M tokens    7.4% of train
supreme_court/val                 2.7M tokens  
tribunals/train                1426.8M tokens   41.9% of train
tribunals/val                    14.0M tokens  
TOTAL train tokens: 3.406 B   (6.8 GB)

--- decoded sample from tribunals_train_0009.bin @ 64238346 ---
 e ls nq?kZVukRed {kfr ugha gksuk ikrs gq, fujLr fd;k gS tks fd izn"kZ lh&1 ,oa vksih&5 ls izekf.kr gSA 09- vihykFkhZ@ifjoknh }kjk izLrqr nkos dks mRrjoknh@fo:) i{kdkj chek daiuh }kjk okgu dh {kfr nq?kZVuk Lo:i u gksdj lkekU; VwV&QwV (Routine Wear & Tear) dh otg ls gS tks fd ikWfylh esa ns; ugha gS] mYysf[kr djrs gq, Dismissed Page 4 of 6 Appeal No. Krishna Kumar Yadav Vs. D.M., United India Ins. Co. Ltd. Order Pronounced FA/2023/216 On- 28.12.2023

In [7]:
if HF_ARTIFACT_REPO and ALL_DONE:
    hf_push(f"shards/{TOKENIZER_NAME}")